# 레슨 01 - AI 에이전트 소개

**초보자를 위한 AI 에이전트** 코스의 첫 번째 레슨에 오신 것을 환영합니다!

<strong>AI 에이전트</strong>는 대형 언어 모델(LLM)을 추론 엔진으로 사용하고 실제 세계에서 <em>행동</em>을 취할 수 있는 프로그램입니다 — API 호출, 데이터베이스 조회 또는 코드 실행 등을 통해 사용자를 대신해 목표를 달성합니다.

이 노트북에서 여러분은 첫 번째 에이전트인 <strong>여행 에이전트</strong>를 만들 것입니다. 이 에이전트는 휴가지 추천을 합니다. 과정 중에 다음을 배우게 됩니다:

1. <strong>Microsoft Agent Framework</strong>를 사용해 Microsoft Foundry Agent Service에 연결하는 법.
2. 에이전트에게 <strong>도구</strong> — 호출 가능한 일반 Python 함수를 제공하는 법.
3. 에이전트를 실행하고 그 응답을 검사하는 법.
4. 에이전트의 응답을 토큰 단위로 스트리밍하는 법.


## 설정

이 노트북을 실행하기 전에 다음이 준비되어 있는지 확인하세요:

1. **배포된 챗 모델이 있는 Microsoft Foundry 프로젝트** (예: `gpt-4.1-mini`).
2. **Azure CLI에 로그인** — 터미널에서 `az login` 명령 실행.
3. **필요한 환경 변수 설정:**
   - `AZURE_AI_PROJECT_ENDPOINT` — 귀하의 Microsoft Foundry 프로젝트 엔드포인트.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 배포한 모델 이름.

아래 셀은 필요한 Python 패키지를 설치합니다.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 처음 에이전트 만들기

에이전트는 두 가지가 필요합니다:

- <strong>지침</strong> — <em>자신이 누구인지</em>와 *어떻게 행동할지* 알려주는 것 (시스템 프롬프트).
- <strong>도구</strong> — 에이전트가 정보를 얻거나 작업을 수행하기 위해 호출할 수 있는 `@tool`로 장식된 Python 함수들.

아래에서는 인기 있는 휴가지 목록을 반환하는 간단한 도구를 정의합니다. 사용자가 여행 추천을 요청할 때 에이전트가 이 도구를 사용할 것입니다.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 스트리밍 응답

보다 인터랙티브한 경험을 위해 에이전트의 응답을 <strong>스트리밍</strong>할 수 있습니다. 전체 응답을 기다리는 대신, 에이전트는 생성하는 대로 텍스트 청크를 전송합니다. 이는 실시간으로 출력을 보여주고자 하는 채팅 인터페이스에서 특히 유용합니다.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## 요약

이 수업에서는 다음을 배웠습니다:

- Microsoft Foundry Agent Service에 `FoundryChatClient`를 통해 연결하는 **프로바이더 생성**.
- 에이전트가 Python 함수를 호출할 수 있도록 `@tool` 데코레이터를 사용해 **도구 정의**.
- 사용자 메시지로 에이전트를 **실행하고 응답을 출력**.
- 실시간 출력을 위한 **응답 스트리밍**.

다음 수업에서는 에이전트 프레임워크를 더 깊이 탐구하고 에이전트에 더 강력한 도구와 다단계 추론 기능을 제공하는 방법을 배울 것입니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
